# GOES Event: 30 March 2026 to 3 April 2026
In this Notebook, I am investigating the energetic particle event observed in the GOES data on 1 April 2026. This figure plots time vs flux from 6:00 on March 30 to 6:00 on April 03, 2026. It plots data from Parker Solar Probe, Solar Orbiter, SOHO STEREO-A and WIND for 100 keV electrons, 1 MeV electrons and >25 MeV protons/ions.

## 100 keV Electron panel
- PSP ISOIS-EPILO F (W3) 65.9 - 153.5 keV
- SOLO/EPT sun 0.0866 - 0.1314 MeV
- STEREO-A/SEPT sun 85-125 keV
- Wind/3DP omni 75.91 - 140.97 keV

## 1 MeV Electron panel
- PSP ISOIS-EPIHI-HET A (sun) 0.71 - 1.41 MeV
- SOLO/HET sun 0.45 - 2.40 MeV
- STEREO-A/HET 0.7 - 2.8 MeV
- SOHO/EPHIN 0.67 - 10.4 MeV

## >25 MeV Protons/Ions panel
- PSP ISOIS-EPHIH-HET A (sun) 26.9 - 38.1 MeV
- SOLO/HET sun 25.09 - 41.18 MeV
- STEREO-A/HET 26.3 - 40.5 MeV
- SOHO/ERNE/HED 25.0 - 40.0 MeV

The data was downloaded using the SERPENTINE data_loader tool (https://serpentine-h2020.eu).

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import cdflib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [3]:
# ============================================================
# Settings
# ============================================================
RES   = "10min"
START = pd.Timestamp("2026-03-30 00:00")
END   = pd.Timestamp("2026-04-02 00:00")   # exclusive end: covers 30 Mar, 31 Mar, 1 Apr

# Replaces the hard-coded DAYS tuple; works across month boundaries
DAYS = pd.date_range(START.normalize(), END - pd.Timedelta(seconds=1), freq="D")
# -> 2026-03-30, 2026-03-31, 2026-04-01

DIR_100KEV = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\100keV_electrons") 
DIR_1MEV   = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\1MeV_electrons")
DIR_PROTON = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\25MeV_protons")

COLOURS = {"PSP": "C0", "SolO": "C1", "STEREO-A": "C2", "SOHO": "C3", "Wind": "C4"}

# Quality-flag masks: (variable, minimum value to keep) or None for no mask
QF_SOLO_EPT = ("QUALITY_FLAG_1", 3)   # your original 100 keV choice
QF_SOLO_HET_ELEC   = None             # e.g. ("QUALITY_FLAG_4", 3) once checked
QF_SOLO_HET_PROTON = None             # e.g. ("QUALITY_FLAG", 3) once checked


In [4]:

# ============================================================
# Shared helpers
# ============================================================
def cdf_times(cdf, var):
    return pd.DatetimeIndex(cdflib.cdfepoch.to_datetime(cdf.varget(var)))

def resample(s):
    s = pd.Series(np.asarray(s, dtype=float), index=s.index).resample(RES).mean()
    return s.where(s > 0)                      # zeros can't go on a log axis

def band_average(flux, widths):
    """NaN-robust width-weighted mean. flux: (n_time, n_bins); widths: (n_bins,)."""
    flux = np.asarray(flux, dtype=float)
    flux = np.where(flux < 0, np.nan, flux)    # fill values (-1e31) and negatives
    valid = np.isfinite(flux)
    w = np.where(valid, np.asarray(widths, dtype=float), 0.0)
    num = np.sum(np.where(valid, flux, 0.0) * w, axis=1)
    den = w.sum(axis=1)
    out = np.full(len(flux), np.nan)
    np.divide(num, den, out=out, where=den > 0)
    return out

def day_files(data_dir, pattern, days=DAYS):
    """One file per day; if several versions exist, take the latest."""
    out = []
    for d in days:
        hits = sorted(data_dir.glob(pattern.format(d=d)))
        if not hits:
            raise FileNotFoundError(f"No match for {pattern.format(d=d)} in {data_dir}")
        out.append(hits[-1])
    return out

def load_band(data_dir, pattern, flux_var, epoch_var, idx, widths, qf=None):
    """Read selected energy bins from daily CDFs -> one width-weighted Series."""
    parts = []
    for f in day_files(data_dir, pattern):
        c = cdflib.CDF(str(f))
        t = cdf_times(c, epoch_var)
        flux = np.asarray(c.varget(flux_var), dtype=float)[:, idx]
        if qf is not None:
            flux[np.asarray(c.varget(qf[0])) < qf[1]] = np.nan
        parts.append(pd.Series(band_average(flux, widths), index=t))
    return pd.concat(parts).sort_index()

In [5]:
# STEREO-A loader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_sept(files):
    bins = [f"{i:02d}" for i in range(2, 17)]
    names = (["jd", "year", "doy", "hour", "minute", "second"]
             + [f"flux_bin{b}" for b in bins]
             + [f"unc_bin{b}" for b in bins]
             + ["accum_time"])
    df = pd.concat(
        [pd.read_csv(f, comment="#", sep=r"\s+", header=None, names=names)
         for f in files],
        ignore_index=True,
    )
    df.index = pd.DatetimeIndex(
        pd.to_datetime(df["jd"], unit="D", origin="julian").dt.round("s"),
        name="time",
    )
    df = df.drop(columns=["jd", "year", "doy", "hour", "minute", "second"])
    df = df.replace([-9999.9, -9999], np.nan)
    return df.sort_index()

DIR_STA_SEPT = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\100keV_electrons")  # folder with the .dat files
DOYS_SEPT    = (89, 90, 91, 92, 93)

sept_files = day_files(
    DIR_STA_SEPT,
    "sept_ahead_ele_sun_2026_{d:03d}_1min_l2_v*.dat",
    days=DOYS_SEPT,
)
sept = load_sept(sept_files)
print(sept.shape)
print(sept.index[0], "->", sept.index[-1])
sept[["flux_bin06", "flux_bin07"]].describe()

(7200, 31)
2026-03-30 00:00:38 -> 2026-04-03 23:59:39


,flux_bin06,flux_bin07
count,7200.000000,7200.000000
mean,1321.203176,817.370722
std,2723.369176,1672.128412
min,0.000000,0.000000
25%,9.805000,0.055823
50%,29.415000,17.435000
75%,460.830000,287.497500
max,10031.000000,6465.800000


In [7]:
# ============================================================
# Panel 1: ~100 keV electrons
# ============================================================
def load_psp_100(files, wedge=3, bins=slice(8, 12)):
    parts = []
    for f in files:
        c = cdflib.CDF(str(f))
        t = cdf_times(c, "Epoch_ChanE")
        flux = c.varget("Electron_Flux_ChanE")[:, wedge, bins]
        width = (c.varget("Electron_ChanE_Energy_DELTAPLUS")[0, wedge, bins]
                 + c.varget("Electron_ChanE_Energy_DELTAMINUS")[0, wedge, bins])
        parts.append(pd.Series(band_average(flux, width) * 1e3, index=t))   # per keV -> per MeV
    return pd.concat(parts).sort_index()

def load_solo_100(files, bins=slice(14, 19), qf=QF_SOLO_EPT):
    parts = []
    for f in files:
        c = cdflib.CDF(str(f))
        t = cdf_times(c, "EPOCH_1")
        flux = c.varget("Electron_Flux")[:, bins].astype(float)
        if qf is not None:
            flux[np.asarray(c.varget(qf[0])) < qf[1]] = np.nan
        width = c.varget("Electron_Bins_Width")[bins]
        parts.append(pd.Series(band_average(flux, width), index=t))         # already per MeV
    return pd.concat(parts).sort_index()

def load_wind_100(files, channel=3):
    parts = []
    for f in files:
        c = cdflib.CDF(str(f))
        tt = c.varget("TIME").astype(float)
        ok = np.isfinite(tt) & (tt > 0)
        flux = c.varget("FLUX")[:, channel].astype(float)
        flux = np.where(flux < 0, np.nan, flux) * 1e6                       # per eV -> per MeV
        parts.append(pd.Series(flux[ok], index=pd.DatetimeIndex(pd.to_datetime(tt[ok], unit="s"))))
    return pd.concat(parts).sort_index()

def load_sept_100(sept, cols=("flux_bin06", "flux_bin07"), widths=(20, 20)):
    return pd.Series(band_average(sept[list(cols)].values, widths), index=sept.index)

def build_100keV(sept):
    psp  = load_psp_100 (day_files(DIR_100KEV, "psp_isois-epilo_l2-pe_{d:%Y%m%d}_v*.cdf"))
    solo = load_solo_100(day_files(DIR_100KEV, "solo_L2_epd-ept-sun-rates_{d:%Y%m%d}_V*.cdf"))
    wind = load_wind_100(day_files(DIR_100KEV, "wi_sfsp_3dp_{d:%Y%m%d}_v*.cdf"))
    sta  = load_sept_100(sept)
    series = {
        "PSP/EPI-Lo W3, 65.9-153.5 keV": (resample(psp),  "PSP"),
        "SolO/EPT sun, 86.6-131.4 keV":  (resample(solo), "SolO"),
        "STEREO-A/SEPT sun, 85-125 keV": (resample(sta),  "STEREO-A"),
        "Wind/3DP omni, 75.9-141.0 keV": (resample(wind), "Wind"),
    }
    events = {
        "STEREO-A": dict(onset="2022-03-28 11:34:51.974", peak_time="2022-03-28 12:28:51.974"),
        "Wind":     dict(onset="2022-03-28 11:46:53.709", peak_time="2022-03-28 12:59:23.549"),
    }
    return series, events

NameError: name 'c' is not defined

In [10]:
from pathlib import Path

data_dir = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\100keV_electrons")   # change to your folder
files = sorted(data_dir.glob("*"))

print(len(files), "files")
for f in files:
    print(f.name)

21 files
psp_isois-epilo_l2-pe_20260330_v26.cdf
psp_isois-epilo_l2-pe_20260331_v26.cdf
psp_isois-epilo_l2-pe_20260401_v26.cdf
psp_isois-epilo_l2-pe_20260402_v26.cdf
psp_isois-epilo_l2-pe_20260403_v26.cdf
sept_ahead_ele_sun_2026_089_1min_l2_v03.dat
sept_ahead_ele_sun_2026_090_1min_l2_v03.dat
sept_ahead_ele_sun_2026_091_1min_l2_v03.dat
sept_ahead_ele_sun_2026_092_1min_l2_v03.dat
sept_ahead_ele_sun_2026_093_1min_l2_v03.dat
solo_L2_epd-het-sun-rates_20260330_V01.cdf
solo_L2_epd-het-sun-rates_20260331_V01.cdf
solo_L2_epd-het-sun-rates_20260401_V01.cdf
solo_L2_epd-het-sun-rates_20260402_V01.cdf
solo_L2_epd-het-sun-rates_20260403_V01.cdf
wi_sfsp_3dp_00000000_v01.cdf
wi_sfsp_3dp_20260330_v01.cdf
wi_sfsp_3dp_20260331_v01.cdf
wi_sfsp_3dp_20260401_v01.cdf
wi_sfsp_3dp_20260402_v01.cdf
wi_sfsp_3dp_20260403_v01.cdf
